# 07 - Previsione del consumo energetico domestico

In questo notebook costruiamo un progetto completo di **time series forecasting** sul dataset **Individual Household Electric Power Consumption** della UCI.

Obiettivo del progetto:

- prevedere il consumo orario `Global_active_power` a partire dalla serie storica minuto per minuto
- rispettare rigorosamente la natura sequenziale dei dati
- confrontare un baseline semplice con modelli `Ridge`, `RandomForestRegressor` e `GradientBoostingRegressor`
- minimizzare il `RMSE`, confrontare il `MAE` con il baseline e puntare a `R2 > 0.80`

La pipeline segue questa struttura:

1. preparazione del dataset e contesto
2. preprocessing e split cronologico
3. feature engineering con feature temporali, codifica ciclica, lag e rolling statistics
4. EDA con profili medi, stagionalita e ACF/PACF
5. modellazione e validazione con `TimeSeriesSplit`
6. valutazione finale ed error analysis
7. salvataggio del modello con `joblib`


## Ambiente consigliato

File richiesto nella cartella corrente:

- `household_power_consumption.csv`

Pacchetti consigliati:

```powershell
python -m pip install pandas numpy matplotlib seaborn scikit-learn statsmodels joblib
```

Il notebook e' preparato per essere eseguito **step by step** e non contiene output precalcolati.


In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

RANDOM_STATE = 42
TARGET_COLUMN = 'Global_active_power'
DATA_PATH = Path.cwd() / 'household_power_consumption.csv'
ARTIFACTS_DIR = Path.cwd() / 'artifacts' / '07_previsione_energia'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'File non trovato: {DATA_PATH}')

print('Dataset:', DATA_PATH.resolve())
print('Cartella artifacts:', ARTIFACTS_DIR.resolve())


## 1. Dataset Preparation & Context

Il dataset contiene misure elettriche domestiche a frequenza minuto per minuto. In questo esercizio usiamo come target `Global_active_power` e lo convertiamo a frequenza oraria con:

- gestione dei valori mancanti (`?` nel file originale)
- parsing temporale con indice `DatetimeIndex`
- interpolazione temporale per colmare i buchi
- aggregazione oraria tramite `.resample('H').mean()`

Per contenere l'uso di memoria leggiamo solo le colonne strettamente necessarie al forecasting.


In [ ]:
raw_df = pd.read_csv(
    DATA_PATH,
    sep=';',
    na_values='?',
    low_memory=False,
    usecols=['Date', 'Time', TARGET_COLUMN],
)

raw_df['timestamp'] = pd.to_datetime(
    raw_df['Date'] + ' ' + raw_df['Time'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce',
)

raw_df = raw_df.drop(columns=['Date', 'Time'])
raw_df[TARGET_COLUMN] = pd.to_numeric(raw_df[TARGET_COLUMN], errors='coerce')
raw_df = raw_df.set_index('timestamp').sort_index()
raw_df = raw_df[~raw_df.index.duplicated(keep='first')]

missing_summary = pd.DataFrame(
    {
        'missing_count': raw_df.isna().sum(),
        'missing_ratio': raw_df.isna().mean(),
    }
)
display(missing_summary)

# Interpolazione temporale al minuto per ottenere una serie continua prima del resampling.
raw_df[TARGET_COLUMN] = raw_df[TARGET_COLUMN].interpolate(
    method='time',
    limit_direction='both',
)

# Aggregazione oraria richiesta dalla traccia.
hourly_df = raw_df[[TARGET_COLUMN]].resample('H').mean()
hourly_df[TARGET_COLUMN] = hourly_df[TARGET_COLUMN].interpolate(
    method='time',
    limit_direction='both',
)

display(hourly_df.head())
display(hourly_df.tail())

print('Numero osservazioni originali:', len(raw_df))
print('Numero osservazioni orarie:', len(hourly_df))
print('Periodo coperto:', hourly_df.index.min(), '->', hourly_df.index.max())
print('Valori mancanti residui:', int(hourly_df[TARGET_COLUMN].isna().sum()))


## 2. Preprocessing & Data Splitting

Qui avviene il passaggio piu' delicato dell'esercizio.

- lo split deve essere **strettamente cronologico**
- il train deve contenere solo osservazioni precedenti al test
- non usiamo `train_test_split(..., shuffle=True)` in nessun punto

Prima dello split costruiamo tutte le feature che dipendono solo dal passato.


## 3. Feature Engineering

Feature richieste dalla traccia:

- feature temporali: `hour`, `day_of_week`, `month`, `is_weekend`
- codifica ciclica per ora e giorno della settimana
- lag: `lag_1h`, `lag_24h`, `lag_168h`
- rolling statistics: `rolling_24h_mean`, `rolling_168h_std`

Nota importante: per evitare leakage, le rolling statistics sono calcolate su una serie shiftata di una posizione, quindi usano solo informazione disponibile prima dell'istante da prevedere.


In [ ]:
def create_time_series_features(data, target_col=TARGET_COLUMN):
    df = data.copy()

    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    df['lag_1h'] = df[target_col].shift(1)
    df['lag_24h'] = df[target_col].shift(24)
    df['lag_168h'] = df[target_col].shift(168)

    shifted_target = df[target_col].shift(1)
    df['rolling_24h_mean'] = shifted_target.rolling(window=24, min_periods=24).mean()
    df['rolling_168h_std'] = shifted_target.rolling(window=168, min_periods=168).std()

    return df.dropna().copy()


model_df = create_time_series_features(hourly_df)

FEATURE_COLUMNS = [
    'hour',
    'day_of_week',
    'month',
    'is_weekend',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'lag_1h',
    'lag_24h',
    'lag_168h',
    'rolling_24h_mean',
    'rolling_168h_std',
]

split_index = int(len(model_df) * 0.80)
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

assert train_df.index.max() < test_df.index.min(), 'Lo split cronologico non e valido.'

print('Numero righe finali dopo lag e rolling:', len(model_df))
print('Shape train:', X_train.shape, y_train.shape)
print('Shape test:', X_test.shape, y_test.shape)
print('Train range:', train_df.index.min(), '->', train_df.index.max())
print('Test range:', test_df.index.min(), '->', test_df.index.max())

display(model_df[FEATURE_COLUMNS + [TARGET_COLUMN]].head())


## 4. Exploratory Data Analysis (EDA)

Visualizziamo:

- profilo medio giornaliero
- profilo medio settimanale
- stagionalita complessiva tramite media mensile
- autocorrelazione con `ACF` e `PACF`


In [ ]:
daily_profile = hourly_df.assign(hour=hourly_df.index.hour).groupby('hour')[TARGET_COLUMN].mean()
weekly_profile = hourly_df.assign(day_of_week=hourly_df.index.dayofweek).groupby('day_of_week')[TARGET_COLUMN].mean()
monthly_trend = hourly_df[TARGET_COLUMN].resample('M').mean()

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

daily_profile.plot(ax=axes[0], marker='o', color='tab:blue')
axes[0].set_title('Profilo medio giornaliero')
axes[0].set_xlabel('Ora del giorno')
axes[0].set_ylabel('Consumo medio (kW)')

axes[1].plot(day_labels, weekly_profile.reindex(range(7)).values, marker='o', color='tab:green')
axes[1].set_title('Profilo medio settimanale')
axes[1].set_xlabel('Giorno della settimana')
axes[1].set_ylabel('Consumo medio (kW)')

monthly_trend.plot(ax=axes[2], color='tab:orange')
axes[2].set_title('Stagionalita complessiva (media mensile)')
axes[2].set_xlabel('Tempo')
axes[2].set_ylabel('Consumo medio (kW)')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

plot_acf(hourly_df[TARGET_COLUMN].dropna(), lags=168, ax=axes[0])
axes[0].set_title('ACF fino a 168 lag orari')

plot_pacf(hourly_df[TARGET_COLUMN].dropna(), lags=168, ax=axes[1], method='ywm')
axes[1].set_title('PACF fino a 168 lag orari')

plt.tight_layout()
plt.show()


## 5. Modeling & Cross-Validation

Confrontiamo:

- baseline naive che predice il valore dell'ora precedente
- `Ridge`
- `RandomForestRegressor`
- `GradientBoostingRegressor`

La validazione usa `TimeSeriesSplit`, quindi ogni fold rispetta il verso del tempo.


In [ ]:
class Lag1BaselineRegressor(BaseEstimator, RegressorMixin):
    def fit(self, X, y=None):
        if 'lag_1h' not in X.columns:
            raise ValueError('La feature lag_1h e necessaria per il baseline.')
        return self

    def predict(self, X):
        return X['lag_1h'].to_numpy()


def rmse_metric(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


def regression_metrics(y_true, y_pred):
    return {
        'RMSE': mean_squared_error(y_true, y_pred, squared=False),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }


rmse_scorer = make_scorer(rmse_metric, greater_is_better=False)
tscv = TimeSeriesSplit(n_splits=5)
scoring = {
    'rmse': rmse_scorer,
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2',
}

models = {
    'Baseline_lag_1h': Lag1BaselineRegressor(),
    'Ridge': Pipeline(
        steps=[
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=3.0)),
        ]
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=RANDOM_STATE,
    ),
}

cv_rows = []

for name, model in models.items():
    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=tscv,
        scoring=scoring,
        return_train_score=False,
        error_score='raise',
    )

    cv_rows.append(
        {
            'model': name,
            'cv_rmse_mean': -scores['test_rmse'].mean(),
            'cv_rmse_std': (-scores['test_rmse']).std(),
            'cv_mae_mean': -scores['test_mae'].mean(),
            'cv_r2_mean': scores['test_r2'].mean(),
        }
    )

cv_results_df = pd.DataFrame(cv_rows).sort_values('cv_rmse_mean').reset_index(drop=True)
display(cv_results_df)

fitted_models = {}
holdout_rows = []

for name, model in models.items():
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)
    predictions = fitted_model.predict(X_test)

    fitted_models[name] = fitted_model
    metrics = regression_metrics(y_test, predictions)

    holdout_rows.append(
        {
            'model': name,
            'test_rmse': metrics['RMSE'],
            'test_mae': metrics['MAE'],
            'test_r2': metrics['R2'],
        }
    )

holdout_results_df = pd.DataFrame(holdout_rows).sort_values('test_rmse').reset_index(drop=True)
comparison_df = cv_results_df.merge(holdout_results_df, on='model')
display(comparison_df.sort_values('test_rmse').reset_index(drop=True))

# Per la fase finale scegliamo il miglior modello ML, tenendo il baseline solo come riferimento.
ml_ranking = cv_results_df[cv_results_df['model'] != 'Baseline_lag_1h'].reset_index(drop=True)
best_model_name = ml_ranking.iloc[0]['model']
best_model = fitted_models[best_model_name]
best_predictions = best_model.predict(X_test)
best_metrics = regression_metrics(y_test, best_predictions)

baseline_mae = holdout_results_df.loc[
    holdout_results_df['model'] == 'Baseline_lag_1h',
    'test_mae',
].iloc[0]

print('Miglior modello ML per CV RMSE:', best_model_name)
print('RMSE holdout:', round(best_metrics['RMSE'], 4))
print('MAE holdout:', round(best_metrics['MAE'], 4))
print('R2 holdout:', round(best_metrics['R2'], 4))
print('MAE baseline:', round(baseline_mae, 4))
print('Delta MAE vs baseline:', round(baseline_mae - best_metrics['MAE'], 4))
print('Target R2 > 0.80 raggiunto:', best_metrics['R2'] > 0.80)


## 6. Evaluation & Error Analysis

Per il miglior modello ML analizziamo:

- residui nel tempo
- comportamento medio per blocchi orari
- scatter plot `actual vs predicted`

## 7. Deployment

Alla fine salviamo il modello con `joblib` insieme alle feature utilizzate e agli intervalli temporali del train/test set.


In [ ]:
diagnostics_df = pd.DataFrame(
    {
        'actual': y_test,
        'predicted': best_predictions,
    },
    index=y_test.index,
)
diagnostics_df['residual'] = diagnostics_df['actual'] - diagnostics_df['predicted']
diagnostics_df['abs_error'] = diagnostics_df['residual'].abs()
diagnostics_df['squared_error'] = diagnostics_df['residual'] ** 2
diagnostics_df['hour'] = diagnostics_df.index.hour

display(diagnostics_df.head())

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

diagnostics_df['residual'].plot(ax=axes[0], color='tab:red')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title(f'Residui nel tempo - {best_model_name}')
axes[0].set_ylabel('Errore (kW)')

diagnostics_df[['actual', 'predicted']].iloc[:24 * 14].plot(ax=axes[1])
axes[1].set_title('Prime due settimane del test set: reale vs predetto')
axes[1].set_ylabel('kW')
axes[1].set_xlabel('Timestamp')

plt.tight_layout()
plt.show()

hourly_error_df = diagnostics_df.groupby('hour').agg(
    actual_mean=('actual', 'mean'),
    predicted_mean=('predicted', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda values: np.sqrt(np.mean(values))),
).reset_index()

display(hourly_error_df)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].plot(hourly_error_df['hour'], hourly_error_df['actual_mean'], marker='o', label='Actual')
axes[0].plot(hourly_error_df['hour'], hourly_error_df['predicted_mean'], marker='o', label='Predicted')
axes[0].set_title('Prestazioni medie per ora del giorno')
axes[0].set_xlabel('Ora')
axes[0].set_ylabel('Consumo medio (kW)')
axes[0].legend()

sns.barplot(data=hourly_error_df, x='hour', y='mae', ax=axes[1], color='tab:blue')
axes[1].set_title('MAE per blocco orario')
axes[1].set_xlabel('Ora')
axes[1].set_ylabel('MAE (kW)')

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 7))
sns.scatterplot(data=diagnostics_df, x='actual', y='predicted', alpha=0.35)
limit_min = min(diagnostics_df['actual'].min(), diagnostics_df['predicted'].min())
limit_max = max(diagnostics_df['actual'].max(), diagnostics_df['predicted'].max())
plt.plot([limit_min, limit_max], [limit_min, limit_max], linestyle='--', color='black')
plt.title(f'Actual vs Predicted - {best_model_name}')
plt.xlabel('Actual (kW)')
plt.ylabel('Predicted (kW)')
plt.tight_layout()
plt.show()

MODEL_PATH = ARTIFACTS_DIR / 'best_energy_consumption_model.joblib'

artifact = {
    'model_name': best_model_name,
    'model': best_model,
    'feature_columns': FEATURE_COLUMNS,
    'target_column': TARGET_COLUMN,
    'train_range': (str(train_df.index.min()), str(train_df.index.max())),
    'test_range': (str(test_df.index.min()), str(test_df.index.max())),
    'hourly_frequency': 'H',
}

joblib.dump(artifact, MODEL_PATH)
print('Modello salvato in:', MODEL_PATH.resolve())
